# Build an eval set & score a prompt

**Session 3 · small model (`llama3.2:3b`) — the pivot**

Stop eyeballing the classifier. Build a fixed labelled set, score the prompt on it with
`repeats` so you can see the run-to-run spread, then change ONE thing and let `compare()`
say whether the move beat the noise.

Two iterations, two different outcomes:
1. Fix the **output format** — a large, unambiguous win (the first score is low because you
   can't parse the answer, not because the model is wrong).
2. Add label **definitions** — a change that lands *inside* the noise. That is also a result.

We use the **support-ticket router** from Session 2 (the 3B model is ~99% on the sentiment
set, so there is nothing to iterate on there).

In [1]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from utils import ask, SMALL_MODEL
from eval import load_cases, run_eval, print_report, compare, exact


In [8]:
LABELS = ["billing", "bug", "other"]

NAIVE = 'Classify this support ticket into billing, bug, or other.\n\n"{t}"'
cases = load_cases("../eval/datasets/support_tickets.jsonl")

def raw_classifier(prompt_tmpl):
    def classify(text):
        return ask(prompt_tmpl.format(t=text), model=SMALL_MODEL).strip().lower().strip('."\' ')
    return classify


### Score the first prompt — strictly

Score the **raw** model output against the gold label with `exact` (no cleanup). This is the
honest number: if your pipeline can't parse it, it's wrong.

In [ ]:



report = run_eval(cases, raw_classifier(NAIVE), scorer=exact, repeats=3)
print_report(report)

accuracy: 0% +/- 0%  (min 0%, max 0%, 3 runs, n=40)  in 374.52s
  FAIL  in='I was charged twice this month'  exp='billing'  got='i would classify this support ticket as a "billing" issue'
  FAIL  in='The app crashes when I tap export'  exp='bug'  got='i would classify this support ticket as a "bug". the issue is with the app\'s functionality (exporting data) and it\'s causing a crash, which is a technical problem that needs to be fixed'
  FAIL  in='Do you offer a student discount?'  exp='other'  got='i would classify this support ticket as "other". the question is inquiring about a specific policy or feature, rather than reporting a problem or issue with a product or service'
  FAIL  in='Payment went through but my plan still says free'  exp='billing'  got='i would classify this support ticket as "billing". the issue is related to the billing status of the user\'s plan, which is not matching the expected outcome'
  FAIL  in='Export runs but the PDF comes out blank'  exp='bug'  got='i w

### Iteration 1 — fix the output format

The model knows the answer; it just won't say it in one word. Add a format rule and three
examples. Score the same way and `compare()`.

### Worked example

`NAIVE` vs `DISCIPLINED` (format rule + few-shot), strict scoring, then a per-case report of
what still fails.

In [9]:
DISCIPLINED = (
    'Classify the support ticket as billing, bug, or other.\n'
    'Reply with ONE lowercase word and nothing else.\n\n'
    'Ticket: "I cannot log in since the update" -> bug\n'
    'Ticket: "Refund me for the double charge" -> billing\n'
    'Ticket: "What are your office hours?" -> other\n\n'
    'Ticket: "{t}" ->'
)

compare(
    cases,
    raw_classifier(NAIVE),
    raw_classifier(DISCIPLINED),
    labels=("naive", "format rule + few-shot"),
    scorer=exact,
    repeats=3,
)
print()
print_report(run_eval(cases, raw_classifier(DISCIPLINED), scorer=exact, repeats=1))


  naive                      0%   (spread 0% over 3 runs)
  format rule + few-shot     50%   (spread 0% over 3 runs)
  gap +50%   vs   run-to-run noise 0%   ->   REAL

accuracy: 50%  (20/40)  in 15.19s
  FAIL  in='Payment went through but my plan still says free'  exp='billing'  got='bug'
  FAIL  in='How do I change my email address?'  exp='other'  got='bug'
  FAIL  in='My invoice shows the wrong VAT rate'  exp='billing'  got='bug'
  FAIL  in='Is there an API rate limit?'  exp='other'  got='bug'
  FAIL  in='You charged me again after I cancelled'  exp='billing'  got='bug'
  FAIL  in='Can I export my data to CSV?'  exp='other'  got='bug'
  FAIL  in='Which browsers do you support?'  exp='other'  got='bug'
  FAIL  in='My card expired and now I cannot log in to pay'  exp='billing'  got='bug'
  FAIL  in='Where can I find the keyboard shortcuts?'  exp='other'  got='bug'
  FAIL  in='Please cancel my subscription and stop billing me'  exp='billing'  got='bug'
  FAIL  in='How do I invite a team

### Iteration 2 — add label definitions

Switch to a tolerant scorer (`label_in`: the label appears anywhere in the reply) so we're
measuring *accuracy* now, not format. Does spelling out what each label means help?

In [10]:
def label_in(output, expected):
    return expected in output.lower()

WITH_DEFS = (
    'Classify the support ticket as billing, bug, or other.\n'
    '- billing: charges, invoices, receipts, refunds, payment methods, cancellations\n'
    '- bug: a feature is broken, wrong, or behaves unexpectedly\n'
    '- other: how-to questions, account settings, general product questions\n'
    'Reply with ONE lowercase word and nothing else.\n\n'
    'Ticket: "I cannot log in since the update" -> bug\n'
    'Ticket: "Refund me for the double charge" -> billing\n'
    'Ticket: "What are your office hours?" -> other\n\n'
    'Ticket: "{t}" ->'
)

compare(
    cases,
    raw_classifier(DISCIPLINED),
    raw_classifier(WITH_DEFS),
    labels=("no definitions", "with definitions"),
    scorer=label_in,
    repeats=5,
)


  no definitions             50%   (spread 0% over 5 runs)
  with definitions           72%   (spread 0% over 5 runs)
  gap +22%   vs   run-to-run noise 0%   ->   REAL


{'a': {'n': 40,
  'repeats': 5,
  'accuracies': [0.5, 0.5, 0.5, 0.5, 0.5],
  'acc_mean': 0.5,
  'acc_min': 0.5,
  'acc_max': 0.5,
  'spread': 0.0,
  'accuracy': 0.5,
  'passed': 20,
  'results': [{'input': 'I was charged twice this month',
    'expected': 'billing',
    'output': 'billing',
    'pass': True},
   {'input': 'The app crashes when I tap export',
    'expected': 'bug',
    'output': 'bug',
    'pass': True},
   {'input': 'Do you offer a student discount?',
    'expected': 'other',
    'output': 'other',
    'pass': True},
   {'input': 'Payment went through but my plan still says free',
    'expected': 'billing',
    'output': 'bug',
    'pass': False},
   {'input': 'Export runs but the PDF comes out blank',
    'expected': 'bug',
    'output': 'bug',
    'pass': True},
   {'input': 'How do I change my email address?',
    'expected': 'other',
    'output': 'bug',
    'pass': False},
   {'input': 'My invoice shows the wrong VAT rate',
    'expected': 'billing',
    'output':

## Your turn - vary the example

1. Look at the FAIL lines. Add ONE few-shot example targeting the most common failure, re-run
   `compare()`. Did overall accuracy rise past the noise, or did you trade one failure for another?
2. If `compare()` says INCONCLUSIVE, bump `repeats` to 8. Does the verdict firm up, or is the
   change genuinely too small to matter?
3. Record the before/after mean, the spread, and the verdict in your commit message and PR.
